# Scania APS Failure - Viability Notebook

Este notebook desarrolla la base analitica del proyecto de grado sobre mantenimiento predictivo con el dataset `APS Failure at Scania Trucks`.

Objetivo del notebook:
- caracterizar el problema y sus retos de datos
- comparar modelos baseline y competitivos
- evaluar el efecto del costo de error
- seleccionar un umbral de decision para priorizacion
- dejar artefactos reproducibles que alimenten el informe del proyecto


## Bloque 1. Librerias

Cargamos herramientas para exploracion, modelado tabular y evaluacion en problemas desbalanceados.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


## Bloque 2. Configuracion del notebook

Definimos rutas, carpetas de salida y archivos esperados. El notebook puede usar tanto el conjunto de entrenamiento como el conjunto de prueba oficial de Scania.


In [ ]:
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()

TRAIN_CANDIDATE_PATHS = [
    PROJECT_ROOT.parent / 'aps_failure_training_set.csv',
    PROJECT_ROOT / 'data' / 'raw' / 'aps_failure_training_set.csv',
]
TEST_CANDIDATE_PATHS = [
    PROJECT_ROOT.parent / 'aps_failure_test_set.csv',
    PROJECT_ROOT / 'data' / 'raw' / 'aps_failure_test_set.csv',
]

TARGET_COLUMN = 'class'
RANDOM_STATE = 42
FALSE_POSITIVE_COST = 10
FALSE_NEGATIVE_COST = 500

REPORTS_DIR = PROJECT_ROOT / 'reports' / 'scania'
FIGURES_DIR = REPORTS_DIR / 'figures'
TABLES_DIR = REPORTS_DIR / 'tables'
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_DATASET_PATH = next((path for path in TRAIN_CANDIDATE_PATHS if path.exists()), None)
TEST_DATASET_PATH = next((path for path in TEST_CANDIDATE_PATHS if path.exists()), None)

TRAIN_DATASET_PATH, TEST_DATASET_PATH

## Bloque 3. Utilidades de carga y evaluacion

Centralizamos funciones utiles para que el notebook sea mas limpio y reproducible.


In [ ]:
def load_scania_csv(path: Path) -> pd.DataFrame:
    df = pd.read_csv(
        path,
        na_values='na',
        skiprows=19,
        low_memory=False,
    )
    df.columns = df.columns.str.strip()
    return df


def cost_from_confusion(cm, false_positive_cost=FALSE_POSITIVE_COST, false_negative_cost=FALSE_NEGATIVE_COST):
    tn, fp, fn, tp = cm.ravel()
    return fp * false_positive_cost + fn * false_negative_cost


def summarize_predictions(y_true, predictions, positive_label='pos'):
    cm = confusion_matrix(y_true, predictions, labels=['neg', 'pos'])
    return {
        'accuracy': accuracy_score(y_true, predictions),
        'f1_macro': f1_score(y_true, predictions, average='macro'),
        'precision_pos': precision_score(y_true, predictions, pos_label=positive_label, zero_division=0),
        'recall_pos': recall_score(y_true, predictions, pos_label=positive_label, zero_division=0),
        'f1_pos': f1_score(y_true, predictions, pos_label=positive_label, zero_division=0),
        'cost': cost_from_confusion(cm),
        'cm': cm,
    }


## Bloque 4. Carga robusta del dataset de entrenamiento

El archivo de Scania trae un bloque de texto legal antes de la tabla. La funcion de carga ya tiene en cuenta ese detalle.


In [ ]:
if TRAIN_DATASET_PATH is None:
    raise FileNotFoundError('No se encontro aps_failure_training_set.csv en las rutas esperadas.')

df = load_scania_csv(TRAIN_DATASET_PATH)
df.shape

## Bloque 5. Verificacion del encabezado

Confirmamos que la columna objetivo se cargo correctamente.


In [ ]:
df.columns[:5]

In [ ]:
df.head()

## Bloque 6. Revision de faltantes

El dataset se caracteriza por una cantidad considerable de datos faltantes. Eso aumenta el valor metodologico del problema.


In [ ]:
missing_summary = df.isna().mean().sort_values(ascending=False)
missing_top15 = missing_summary.head(15)
missing_top15.to_csv(TABLES_DIR / 'missing_top15.csv', header=['missing_ratio'])
missing_top15

## Bloque 7. Desbalance de clases

Este bloque muestra por que no basta con mirar `accuracy`.


In [ ]:
class_counts = df[TARGET_COLUMN].value_counts(dropna=False)
class_proportions = df[TARGET_COLUMN].value_counts(normalize=True, dropna=False).round(4)
class_summary = pd.DataFrame({'count': class_counts, 'proportion': class_proportions})
class_summary.to_csv(TABLES_DIR / 'class_distribution.csv')
display(class_counts.to_frame('count'))
display(class_proportions.to_frame('proportion'))

## Bloque 8. Preparacion de variables

Separamos predictores y variable objetivo y construimos una particion estratificada para evaluacion interna.


In [ ]:
X = df.drop(columns=[TARGET_COLUMN]).copy()
y = df[TARGET_COLUMN].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

X_train.shape, X_test.shape

## Bloque 9. Baseline trivial

Una referencia que siempre predice la clase mayoritaria.


In [ ]:
dummy_model = DummyClassifier(strategy='most_frequent')
dummy_model.fit(X_train, y_train)
dummy_predictions = dummy_model.predict(X_test)

print('Dummy baseline')
print(f'Accuracy: {accuracy_score(y_test, dummy_predictions):.4f}')
print(f'F1 macro: {f1_score(y_test, dummy_predictions, average="macro"):.4f}')
print(confusion_matrix(y_test, dummy_predictions))
print(classification_report(y_test, dummy_predictions, zero_division=0))

## Bloque 10. Regresion logistica como baseline interpretable

Usamos imputacion por mediana, escalamiento y balanceo por clase.


In [ ]:
logistic_model = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('classifier', LogisticRegression(max_iter=1000, class_weight='balanced')),
    ]
)

logistic_model.fit(X_train, y_train)

## Bloque 11. Evaluacion de la regresion logistica

Miramos metricas orientadas a la clase positiva.


In [ ]:
logistic_predictions = logistic_model.predict(X_test)
logistic_scores = logistic_model.predict_proba(X_test)[:, list(logistic_model.classes_).index('pos')]

print('Logistic regression baseline')
print(f'Accuracy: {accuracy_score(y_test, logistic_predictions):.4f}')
print(f'F1 macro: {f1_score(y_test, logistic_predictions, average="macro"):.4f}')
print(f'Average precision for pos: {average_precision_score((y_test == "pos").astype(int), logistic_scores):.4f}')
print(confusion_matrix(y_test, logistic_predictions))
print(classification_report(y_test, logistic_predictions, zero_division=0))

## Bloque 12. CatBoost como modelo competitivo

CatBoost suele comportarse bien con datos tabulares y faltantes.


In [ ]:
catboost_model = CatBoostClassifier(
    iterations=300,
    depth=6,
    learning_rate=0.08,
    loss_function='Logloss',
    eval_metric='PRAUC',
    auto_class_weights='Balanced',
    verbose=False,
    random_state=RANDOM_STATE,
)

catboost_model.fit(X_train, y_train)

## Bloque 13. Evaluacion de CatBoost

Comparamos su comportamiento frente a la clase positiva minoritaria.


In [ ]:
catboost_predictions = catboost_model.predict(X_test)
catboost_predictions = pd.Series(catboost_predictions.flatten(), index=y_test.index)
catboost_scores = catboost_model.predict_proba(X_test)[:, list(catboost_model.classes_).index('pos')]

print('CatBoost baseline')
print(f'Accuracy: {accuracy_score(y_test, catboost_predictions):.4f}')
print(f'F1 macro: {f1_score(y_test, catboost_predictions, average="macro"):.4f}')
print(f'Average precision for pos: {average_precision_score((y_test == "pos").astype(int), catboost_scores):.4f}')
print(confusion_matrix(y_test, catboost_predictions))
print(classification_report(y_test, catboost_predictions, zero_division=0))

## Bloque 14. Curva precision-recall para la clase positiva

Esta grafica ayuda a evaluar el modelo como herramienta de priorizacion.


In [ ]:
y_test_pos = (y_test == 'pos').astype(int)

logistic_precision, logistic_recall, _ = precision_recall_curve(y_test_pos, logistic_scores)
catboost_precision, catboost_recall, _ = precision_recall_curve(y_test_pos, catboost_scores)

plt.figure(figsize=(8, 5))
plt.plot(logistic_recall, logistic_precision, label='Logistic regression')
plt.plot(catboost_recall, catboost_precision, label='CatBoost')
plt.xlabel('Recall for pos')
plt.ylabel('Precision for pos')
plt.title('Precision-Recall Curve for APS failure risk')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'precision_recall_curve.png', dpi=300, bbox_inches='tight')
plt.show()

## Bloque 15. Evaluacion por costo de error

Usamos la relacion de costos tipica del dataset: `FP=10` y `FN=500`.


In [ ]:
logistic_cm = confusion_matrix(y_test, logistic_predictions, labels=['neg', 'pos'])
catboost_cm = confusion_matrix(y_test, catboost_predictions, labels=['neg', 'pos'])

print(f'Cost assumption: FP={FALSE_POSITIVE_COST}, FN={FALSE_NEGATIVE_COST}')
print(f'Logistic cost: {cost_from_confusion(logistic_cm):,}')
print(f'CatBoost cost: {cost_from_confusion(catboost_cm):,}')

## Bloque 16. Seleccion de umbral para priorizacion

Probamos varios umbrales para la probabilidad de la clase positiva.


In [ ]:
thresholds = np.arange(0.10, 0.95, 0.05)
threshold_rows = []

for threshold in thresholds:
    threshold_predictions = np.where(catboost_scores >= threshold, 'pos', 'neg')
    summary = summarize_predictions(y_test, threshold_predictions)
    threshold_rows.append(
        {
            'threshold': round(float(threshold), 2),
            'precision_pos': summary['precision_pos'],
            'recall_pos': summary['recall_pos'],
            'f1_pos': summary['f1_pos'],
            'cost': summary['cost'],
        }
    )

threshold_results = pd.DataFrame(threshold_rows)
threshold_results.to_csv(TABLES_DIR / 'threshold_results.csv', index=False)
threshold_results.sort_values(by='cost').reset_index(drop=True).head(10)

## Bloque 17. Mejor umbral segun costo

Seleccionamos el umbral con menor costo bajo la suposicion actual.


In [ ]:
best_threshold_row = threshold_results.sort_values(by='cost').iloc[0]
best_threshold = float(best_threshold_row['threshold'])
best_threshold_row.to_frame(name='value')

## Bloque 18. Costo en funcion del umbral

Visualizamos la sensibilidad del costo frente al umbral de decision.


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(threshold_results['threshold'], threshold_results['cost'], marker='o')
plt.axvline(best_threshold, color='red', linestyle='--', label=f"Best threshold={best_threshold:.2f}")
plt.xlabel('Threshold for pos')
plt.ylabel('Cost')
plt.title('Cost sensitivity to decision threshold')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'cost_vs_threshold.png', dpi=300, bbox_inches='tight')
plt.show()

## Bloque 19. Impacto del mejor umbral frente al umbral por defecto

Comparamos la decision estandar (`0.50`) con la decision optimizada por costo.


In [ ]:
default_threshold_predictions = np.where(catboost_scores >= 0.50, 'pos', 'neg')
best_threshold_predictions = np.where(catboost_scores >= best_threshold, 'pos', 'neg')

default_threshold_summary = summarize_predictions(y_test, default_threshold_predictions)
best_threshold_summary = summarize_predictions(y_test, best_threshold_predictions)

threshold_comparison = pd.DataFrame([
    {
        'decision_rule': 'CatBoost threshold 0.50',
        'threshold': 0.50,
        'accuracy': default_threshold_summary['accuracy'],
        'f1_macro': default_threshold_summary['f1_macro'],
        'precision_pos': default_threshold_summary['precision_pos'],
        'recall_pos': default_threshold_summary['recall_pos'],
        'f1_pos': default_threshold_summary['f1_pos'],
        'cost': default_threshold_summary['cost'],
    },
    {
        'decision_rule': 'CatBoost best threshold',
        'threshold': best_threshold,
        'accuracy': best_threshold_summary['accuracy'],
        'f1_macro': best_threshold_summary['f1_macro'],
        'precision_pos': best_threshold_summary['precision_pos'],
        'recall_pos': best_threshold_summary['recall_pos'],
        'f1_pos': best_threshold_summary['f1_pos'],
        'cost': best_threshold_summary['cost'],
    },
])

threshold_comparison.to_csv(TABLES_DIR / 'threshold_comparison.csv', index=False)
threshold_comparison.round(4)

## Bloque 20. Importancia de variables

Identificamos las variables que mas influyen en CatBoost.


In [ ]:
feature_importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': catboost_model.get_feature_importance(),
}).sort_values(by='importance', ascending=False)

feature_importance.to_csv(TABLES_DIR / 'feature_importance.csv', index=False)
feature_importance.head(15)

In [ ]:
top_features = feature_importance.head(15).sort_values(by='importance')

plt.figure(figsize=(8, 6))
plt.barh(top_features['feature'], top_features['importance'])
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.title('Top 15 feature importances - CatBoost')
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'top_feature_importances.png', dpi=300, bbox_inches='tight')
plt.show()

## Bloque 21. Matrices de confusion visuales

Estas figuras comunican visualmente el comportamiento de los modelos.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
ConfusionMatrixDisplay(logistic_cm, display_labels=['neg', 'pos']).plot(ax=axes[0], colorbar=False)
axes[0].set_title('Logistic Regression')
ConfusionMatrixDisplay(catboost_cm, display_labels=['neg', 'pos']).plot(ax=axes[1], colorbar=False)
axes[1].set_title('CatBoost threshold 0.50')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'confusion_matrices_internal.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(best_threshold_summary['cm'], display_labels=['neg', 'pos']).plot(ax=ax, colorbar=False)
ax.set_title(f'CatBoost best threshold ({best_threshold:.2f})')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'confusion_matrix_best_threshold.png', dpi=300, bbox_inches='tight')
plt.show()

## Bloque 22. Resumen comparativo de modelos

Esta tabla resume los resultados principales sobre la particion interna.


In [ ]:
summary_table = pd.DataFrame([
    {
        'model': 'Dummy',
        'accuracy': accuracy_score(y_test, dummy_predictions),
        'f1_macro': f1_score(y_test, dummy_predictions, average='macro'),
        'precision_pos': precision_score(y_test, dummy_predictions, pos_label='pos', zero_division=0),
        'recall_pos': recall_score(y_test, dummy_predictions, pos_label='pos', zero_division=0),
        'f1_pos': f1_score(y_test, dummy_predictions, pos_label='pos', zero_division=0),
        'avg_precision_pos': None,
        'cost': cost_from_confusion(confusion_matrix(y_test, dummy_predictions, labels=['neg', 'pos'])),
    },
    {
        'model': 'Logistic Regression',
        'accuracy': accuracy_score(y_test, logistic_predictions),
        'f1_macro': f1_score(y_test, logistic_predictions, average='macro'),
        'precision_pos': precision_score(y_test, logistic_predictions, pos_label='pos', zero_division=0),
        'recall_pos': recall_score(y_test, logistic_predictions, pos_label='pos', zero_division=0),
        'f1_pos': f1_score(y_test, logistic_predictions, pos_label='pos', zero_division=0),
        'avg_precision_pos': average_precision_score((y_test == 'pos').astype(int), logistic_scores),
        'cost': cost_from_confusion(logistic_cm),
    },
    {
        'model': 'CatBoost',
        'accuracy': accuracy_score(y_test, catboost_predictions),
        'f1_macro': f1_score(y_test, catboost_predictions, average='macro'),
        'precision_pos': precision_score(y_test, catboost_predictions, pos_label='pos', zero_division=0),
        'recall_pos': recall_score(y_test, catboost_predictions, pos_label='pos', zero_division=0),
        'f1_pos': f1_score(y_test, catboost_predictions, pos_label='pos', zero_division=0),
        'avg_precision_pos': average_precision_score((y_test == 'pos').astype(int), catboost_scores),
        'cost': cost_from_confusion(catboost_cm),
    },
])

summary_table.to_csv(TABLES_DIR / 'model_summary.csv', index=False)
summary_table.round(4)

## Bloque 23. Validacion con el test set oficial

Si el archivo oficial de prueba esta disponible, evaluamos el modelo final con el umbral optimizado. Esto fortalece el proyecto porque añade una validacion adicional sobre un conjunto independiente.


In [ ]:
official_test_results = None

if TEST_DATASET_PATH is not None:
    official_test_df = load_scania_csv(TEST_DATASET_PATH)
    X_official_test = official_test_df.drop(columns=[TARGET_COLUMN]).copy()
    y_official_test = official_test_df[TARGET_COLUMN].copy()

    official_scores = catboost_model.predict_proba(X_official_test)[:, list(catboost_model.classes_).index('pos')]
    official_predictions_default = np.where(official_scores >= 0.50, 'pos', 'neg')
    official_predictions_best = np.where(official_scores >= best_threshold, 'pos', 'neg')

    official_default_summary = summarize_predictions(y_official_test, official_predictions_default)
    official_best_summary = summarize_predictions(y_official_test, official_predictions_best)

    official_test_results = pd.DataFrame([
        {
            'decision_rule': 'Official test - threshold 0.50',
            'threshold': 0.50,
            'accuracy': official_default_summary['accuracy'],
            'f1_macro': official_default_summary['f1_macro'],
            'precision_pos': official_default_summary['precision_pos'],
            'recall_pos': official_default_summary['recall_pos'],
            'f1_pos': official_default_summary['f1_pos'],
            'cost': official_default_summary['cost'],
        },
        {
            'decision_rule': 'Official test - best threshold',
            'threshold': best_threshold,
            'accuracy': official_best_summary['accuracy'],
            'f1_macro': official_best_summary['f1_macro'],
            'precision_pos': official_best_summary['precision_pos'],
            'recall_pos': official_best_summary['recall_pos'],
            'f1_pos': official_best_summary['f1_pos'],
            'cost': official_best_summary['cost'],
        },
    ])

    official_test_results.to_csv(TABLES_DIR / 'official_test_results.csv', index=False)
    display(official_test_results.round(4))
else:
    print('No official test set was found. This block can be skipped for now.')


## Bloque 24. Conclusiones operativas del notebook

Este bloque resume lo esencial para poder pasar del analisis tecnico al informe del proyecto.


In [ ]:
final_notebook_summary = {
    'selected_model': 'CatBoost',
    'best_threshold_by_cost': round(best_threshold, 2),
    'internal_cost_default_threshold': default_threshold_summary['cost'],
    'internal_cost_best_threshold': best_threshold_summary['cost'],
    'internal_precision_pos_best_threshold': best_threshold_summary['precision_pos'],
    'internal_recall_pos_best_threshold': best_threshold_summary['recall_pos'],
    'internal_f1_pos_best_threshold': best_threshold_summary['f1_pos'],
    'most_important_feature': feature_importance.iloc[0]['feature'],
}

final_summary_df = pd.DataFrame(list(final_notebook_summary.items()), columns=['item', 'value'])
final_summary_df.to_csv(TABLES_DIR / 'final_notebook_summary.csv', index=False)
final_summary_df

## Bloque 25. Lectura de avance del proyecto

Con este notebook ya debe quedar demostrado que:

- el problema es realista y desafiante por el desbalance y los faltantes
- CatBoost supera a los baselines considerados
- el costo de error cambia de forma importante con el umbral
- el mejor umbral permite convertir el modelo en una herramienta de priorizacion
- las variables mas influyentes abren una primera capa de interpretabilidad

A partir de aqui, el proyecto puede avanzar hacia la redaccion formal del informe, la discusion metodologica y, si se desea, la modularizacion del pipeline fuera del notebook.
